# Phase 5 (explanations): phase5_explainability

Scores one saliency method on the models fixed at the phase 4 gate (for the `cnn` and the `rnn`: none,
cVAE and WGAN-GP, best checkpoint plus the other seeds), on validation beats:

| letter | method |
|---|---|
| A | Grad-CAM 1D (convolutional models only) |
| B | integrated gradients |
| C | SmoothGrad |
| D | occlusion |

It needs the grid's checkpoints, which are not on GitHub: attach a dataset holding the extracted Kaggle
zips of the full grid (and the banks) and set `ARTIFACTS_DIR`. This also runs on a laptop after
`python kaggle/ingest_runs.py <zip>` (CPU is fine).

Before running (details in `kaggle/README.md`): Internet on; `GITHUB_TOKEN` secret; inputs `mitdb-raw`
and the artifacts dataset; set the parameters below; Run All; then `git pull` in your clone.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
VARIANTS = ["A"]                      # your letter, or several: ["A", "B", "C", "D"]
ARTIFACTS_DIR = ""                    # the attached dataset with the extracted Kaggle zips of the grid (checkpoints) and the banks
SMOKE_TEST = False                    # True = quick check on fake data (a minute or two, no results)
GIT_REF = "main"                      # branch name or full commit SHA to run
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # the attached mitdb-raw dataset; "" downloads MIT-BIH from PhysioNet
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"
PUSH_RESULTS = True                   # commit the results and push them to GIT_REF (no download or ingest needed)
GIT_NAME = ""                         # your name for the results commit, e.g. "Akash Saha"
GIT_EMAIL = ""                        # your GitHub email for the results commit


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
# Turn the variant letters into config files, e.g. "A" -> configs/phase5_explainability/A_....yaml
configs = []
for letter in VARIANTS:
    (path,) = sorted((REPO / "configs/phase5_explainability").glob(f"{letter}_*.yaml"))
    configs.append(str(path.relative_to(REPO)))
print("configs:", configs)

extra = []
if SMOKE_TEST:
    extra.append("--smoke")
has_raw = RAW_DATASET_DIR and any(Path(RAW_DATASET_DIR).rglob("100.dat"))
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase5_explainability", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs]
if has_raw:
    cmd += ["--raw-dir", RAW_DATASET_DIR]
if ARTIFACTS_DIR:
    cmd += ["--artifacts-dir", ARTIFACTS_DIR]
if PUSH_RESULTS and not SMOKE_TEST:
    branch = GIT_REF if len(GIT_REF) != 40 else "kaggle-results"
    cmd += ["--push-to", branch, "--git-name", GIT_NAME, "--git-email", GIT_EMAIL]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO, env=env).returncode   # env carries the token for the push
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
